# 02 · AdamW, learning-rate schedules, gradient tricks, mixed precision

**Needed for:** GPT-2 (LLM track, step 1) and every training run after it.

A pretraining loop is small, but each piece has a subtle detail that changes results:

| Piece | The detail people get wrong |
|---|---|
| AdamW | Weight decay is *decoupled*: applied to the weights directly, not added to the gradient |
| Warmup + cosine | Adam's second-moment estimate is noisy early on; warmup keeps early steps small |
| Gradient clipping | Clip by the *global* norm across all parameters, not per tensor |
| Gradient accumulation | Divide each micro-batch loss by the number of micro-batches |
| bf16 autocast | Weights and optimizer state stay fp32; only the matmuls run in bf16 |
| Weight tying | The output head and input embedding share one matrix |

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. AdamW from scratch

For each parameter $p$ with gradient $g$, at step $t$ (starting from 1):

$$p \leftarrow p - \eta\lambda p \qquad \text{(decoupled weight decay)}$$
$$m \leftarrow \beta_1 m + (1-\beta_1) g, \qquad v \leftarrow \beta_2 v + (1-\beta_2) g^2$$
$$\hat m = m / (1-\beta_1^t), \qquad \hat v = v / (1-\beta_2^t)$$
$$p \leftarrow p - \eta\, \hat m / (\sqrt{\hat v} + \epsilon)$$

The bias correction matters because $m$ and $v$ start at zero, so early estimates are too small.

In [ ]:
def adamw_step(params, grads, state, lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1):
    """Update params in place. state is a dict holding 't', 'm' (list), 'v' (list)."""
    b1, b2 = betas
    state["t"] = state.get("t", 0) + 1
    t = state["t"]
    if "m" not in state:
        state["m"] = [torch.zeros_like(p) for p in params]
        state["v"] = [torch.zeros_like(p) for p in params]
    with torch.no_grad():
        for p, g, m, v in zip(params, grads, state["m"], state["v"]):
            p.mul_(1 - lr * weight_decay)
            m.mul_(b1).add_(g, alpha=1 - b1)
            v.mul_(b2).addcmul_(g, g, value=1 - b2)
            m_hat = m / (1 - b1**t)
            v_hat = v / (1 - b2**t)
            p.sub_(lr * m_hat / (v_hat.sqrt() + eps))


def test_adamw(fn):
    torch.manual_seed(0)
    w_ref = torch.randn(5, 3, dtype=torch.float64, requires_grad=True)
    w_me = w_ref.detach().clone()
    opt = torch.optim.AdamW([w_ref], lr=1e-2, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1)
    state = {}
    for _ in range(10):
        x = torch.randn(4, 5, dtype=torch.float64)
        opt.zero_grad()
        (x @ w_ref).pow(2).mean().backward()
        g = w_ref.grad.clone()
        opt.step()
        fn([w_me], [g], state, lr=1e-2)
    torch.testing.assert_close(w_me, w_ref.detach())


test_adamw(adamw_step)
print("matches torch.optim.AdamW")

**Which parameters get weight decay?** Common practice (GPT-2/nanoGPT, LLaMA): decay the 2-D weight matrices
(linear layers, embeddings) and *not* biases or norm gains. Decaying a norm gain toward zero shrinks the
whole activation.

In [ ]:
def param_groups(model, weight_decay=0.1):
    decay = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


toy = nn.Sequential(nn.Linear(8, 16), nn.LayerNorm(16), nn.Linear(16, 4))
print([len(g["params"]) for g in param_groups(toy)], "(decay, no decay)")

## 2. Linear warmup, then cosine decay to a floor

In [ ]:
def lr_at(step, max_lr, min_lr, warmup_steps, total_steps):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= total_steps:
        return min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def test_lr(fn):
    kw = dict(max_lr=6e-4, min_lr=6e-5, warmup_steps=100, total_steps=1000)
    assert math.isclose(fn(99, **kw), 6e-4)            # end of warmup reaches max
    assert fn(0, **kw) < fn(50, **kw) < fn(99, **kw)   # rising during warmup
    assert math.isclose(fn(550, **kw), (6e-4 + 6e-5) / 2)  # halfway through cosine
    assert math.isclose(fn(1000, **kw), 6e-5)          # floor after the end
    assert math.isclose(fn(5000, **kw), 6e-5)


test_lr(lr_at)
print([round(lr_at(s, 6e-4, 6e-5, 100, 1000) * 1e4, 2) for s in (0, 50, 99, 300, 550, 800, 999)], "(x1e-4)")

## 3. Clipping by global norm

Compute one norm over *all* gradients together; if it exceeds `max_norm`, scale every gradient by the same
factor. This keeps the update direction, only shortening it. Log the pre-clip norm: spikes predict loss spikes.

In [ ]:
def clip_grad_norm(grads, max_norm):
    total = torch.sqrt(sum(g.pow(2).sum() for g in grads))
    scale = min(1.0, max_norm / (total.item() + 1e-6))
    for g in grads:
        g.mul_(scale)
    return total


def test_clip(fn):
    torch.manual_seed(1)
    a = [torch.randn(10, 10) * 3, torch.randn(7) * 3]
    b = [t.clone() for t in a]
    ps = [nn.Parameter(torch.zeros_like(t)) for t in b]
    for p, t in zip(ps, b):
        p.grad = t
    ref_norm = nn.utils.clip_grad_norm_(ps, max_norm=1.0)
    my_norm = fn(a, 1.0)
    torch.testing.assert_close(my_norm, ref_norm)
    for x, p in zip(a, ps):
        torch.testing.assert_close(x, p.grad)


test_clip(clip_grad_norm)
print("matches nn.utils.clip_grad_norm_")

## 4. Gradient accumulation

To simulate batch size 32 with micro-batches of 8, run 4 forward/backward passes before one optimizer step.
Gradients *add up* across `.backward()` calls, so each micro-batch loss must be divided by 4 to get the
gradient of the mean loss over 32 examples.

(With variable-length sequences, "mean over micro-batch" is not "mean over all tokens". Divide by the total
number of target tokens in the full batch instead. This bug appears in real frameworks.)

In [ ]:
model = nn.Linear(10, 1)
x, y = torch.randn(32, 10), torch.randn(32, 1)

model.zero_grad()
F.mse_loss(model(x), y).backward()
full = model.weight.grad.clone()

model.zero_grad()
accum = 4
for xb, yb in zip(x.chunk(accum), y.chunk(accum)):
    (F.mse_loss(model(xb), yb) / accum).backward()
torch.testing.assert_close(model.weight.grad, full)
print("accumulated gradient == full-batch gradient")

## 5. Mixed precision

| Format | Exponent bits | Mantissa bits | Range | Notes |
|---|---|---|---|---|
| fp32 | 8 | 23 | ~1e38 | master weights, optimizer state |
| fp16 | 5 | 10 | ~6.5e4 | needs loss scaling (small gradients underflow) |
| bf16 | 8 | 7 | ~1e38 | same range as fp32, so no loss scaling; less precise |

bf16 has only 8 bits of precision (about 2–3 decimal digits). That's why weights stay in fp32: an update of
`1e-4` to a weight of `1.0` vanishes entirely in bf16.

In [ ]:
one = torch.tensor(1.0, dtype=torch.bfloat16)
print("1 + 1e-3 in bf16:", (one + 1e-3).item(), "| in fp32:", (torch.tensor(1.0) + 1e-3).item())
print("fp16 max:", torch.finfo(torch.float16).max, "| bf16 max:", torch.finfo(torch.bfloat16).max)
print("smallest normal fp16:", torch.finfo(torch.float16).tiny, "(gradients below this lose precision)")

In [ ]:
net = nn.Sequential(nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 64)).to(device)
inp = torch.randn(8, 64, device=device)
with torch.autocast(device_type=device, dtype=torch.bfloat16):
    out = net(inp)
    loss = out.float().pow(2).mean()   # compute the loss in fp32
loss.backward()
print("activation dtype:", out.dtype, "| weight dtype:", net[0].weight.dtype, "| grad dtype:", net[0].weight.grad.dtype)

With fp16 instead, wrap the step in a `torch.amp.GradScaler`: multiply the loss by a large factor before
backward so small gradients don't underflow, unscale before clipping, and skip steps whose gradients hit inf.

## 6. Weight tying

GPT-2 uses the same matrix for the token embedding (vocab → d) and the output projection (d → vocab).
With vocab 50257 and d 768, that saves 38.6M parameters, about 30% of GPT-2 small.

In [ ]:
V, D = 50257, 768
emb = nn.Embedding(V, D)
head = nn.Linear(D, V, bias=False)
head.weight = emb.weight             # same Parameter object: one tensor, two uses
tied = nn.ModuleList([emb, head])
print("params:", sum(p.numel() for p in tied.parameters()) / 1e6, "M (parameters() de-duplicates shared tensors)")
assert head.weight.data_ptr() == emb.weight.data_ptr()

## Drill

In [ ]:
def my_adamw_step(params, grads, state, lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1):
    raise NotImplementedError


def my_lr_at(step, max_lr, min_lr, warmup_steps, total_steps):
    raise NotImplementedError


def my_clip_grad_norm(grads, max_norm):
    raise NotImplementedError


drill(test_adamw, my_adamw_step)
drill(test_lr, my_lr_at)
drill(test_clip, my_clip_grad_norm)